In [ ]:
import sys
sys.path.append('/kaggle/input/datasets/ikutalila/vits-models')

In [ ]:
import math
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
from scipy.io.wavfile import read
from IPython.display import Audio, display
from tqdm.auto import tqdm, trange

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from torch.cuda.amp import autocast, GradScaler
from torch.nn import AvgPool1d, Conv1d, Conv2d, ConvTranspose1d
from torch.nn.utils import weight_norm
from torch.utils.data import DataLoader

import monotonic_align
from flow import ResidualCouplingBlock
from mel_process import mel_spectrogram_torch, spec_to_mel_torch
from modules import *
from util import *

In [ ]:
data_path = '/kaggle/input/datasets/mathurinache/the-lj-speech-dataset/LJSpeech-1.1'
wavs_path = data_path + "/wavs/"
metadata_path = data_path + "/metadata.csv"
metadata_df = pd.read_csv(metadata_path, sep="|", header=None, quoting=3)

metadata_df.columns = ["file_name", "transcription", "normalized_transcription"]
metadata_df = metadata_df[["file_name", "normalized_transcription"]]
metadata_df.head()

In [ ]:
_pad        = '_'
_punctuation = ';:,.!?¡¿—…"«»“”ü ()-' + "'" + "[]" 
_letters = 'ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz'
symbols = [_pad] + list(_punctuation) + list(_letters)

# Mappings from symbol to numeric ID and vice versa:
_symbol_to_id = {s: i for i, s in enumerate(symbols)}
_id_to_symbol = {i: s for i, s in enumerate(symbols)}

def text_to_sequence(text):
    sequence = []
    for symbol in text:
        symbol_id = _symbol_to_id.get(symbol, None)
        if symbol_id is not None:
            sequence.append(symbol_id)
    return sequence

In [ ]:
mel_basis = {}
hann_window = {}

def spectrogram_torch(y, n_fft, sampling_rate, hop_size, win_size, center=False):
    if torch.min(y) < -1.:
        print('min value is ', torch.min(y))
    if torch.max(y) > 1.:
        print('max value is ', torch.max(y))

    global hann_window
    dtype_device = str(y.dtype) + '_' + str(y.device)
    wnsize_dtype_device = str(win_size) + '_' + dtype_device
    if wnsize_dtype_device not in hann_window:
        hann_window[wnsize_dtype_device] = torch.hann_window(win_size).to(dtype=y.dtype, device=y.device)

    y = torch.nn.functional.pad(y.unsqueeze(1), (int((n_fft-hop_size)/2), int((n_fft-hop_size)/2)), mode='reflect')
    y = y.squeeze(1)

    spec = torch.stft(y, n_fft, hop_length=hop_size, win_length=win_size, window=hann_window[wnsize_dtype_device],
                      center=center, pad_mode='reflect', normalized=False, onesided=True, return_complex = False)
    spec = torch.sqrt(spec.pow(2).sum(-1) + 1e-6)
    return spec

In [ ]:
class TextAudioCollate():
    def __init__(self, return_ids=False):
        self.return_ids = return_ids

    def __call__(self, batch):
        _, ids_sorted_decreasing = torch.sort(
            torch.LongTensor([x[1].size(1) for x in batch]),
            dim=0, descending=True)
        
        max_text_len = max([len(x[0]) for x in batch])
        max_spec_len = max([x[1].size(1) for x in batch])
        max_wav_len = max([x[2].size(1) for x in batch])
        
        text_lengths = torch.LongTensor(len(batch))
        spec_lengths = torch.LongTensor(len(batch))
        wav_lengths = torch.LongTensor(len(batch))

        text_padded = torch.LongTensor(len(batch), max_text_len)
        spec_padded = torch.FloatTensor(len(batch), batch[0][1].size(0), max_spec_len)
        wav_padded = torch.FloatTensor(len(batch), 1, max_wav_len)
        text_padded.zero_()
        spec_padded.zero_()
        wav_padded.zero_()
        
        for i in range(len(ids_sorted_decreasing)):
            row = batch[ids_sorted_decreasing[i]]

            text = row[0]
            text_padded[i, :text.size(0)] = text
            text_lengths[i] = text.size(0)

            spec = row[1]
            spec_padded[i, :, :spec.size(1)] = spec
            spec_lengths[i] = spec.size(1)

            wav = row[2]
            wav_padded[i, :, :wav.size(1)] = wav
            wav_lengths[i] = wav.size(1)

        if self.return_ids:
            return text_padded, text_lengths, spec_padded, spec_lengths, wav_padded, wav_lengths, ids_sorted_decreasing
        return text_padded, text_lengths, spec_padded, spec_lengths, wav_padded, wav_lengths

In [ ]:
class TextAudioLoader(torch.utils.data.Dataset):
    def __init__(self, audiopaths_and_text):
        self.audio_path = audiopaths_and_text['file_name'].values
        self.text = audiopaths_and_text['normalized_transcription'].values
        self.sampling_rate = 22050

        self.max_wav_value = 32768.0
        self.filter_length = 1024
        self.hop_length = 256
        self.win_length = 1024
        self._filter()
        
    def _filter(self):
        audios = []
        texts = []
        
        for audiopath, text in zip(self.audio_path, self.text):
            if 1 <= len(text) and len(text) <= 100:
                audios.append(audiopath)
                texts.append(text) 
        self.audio_path =  audios
        self.text = texts
        
    def get_audio(self, filename):
        full_path = data_path + '/wavs/'+ filename + '.wav'
        sampling_rate, audio = read(full_path)
        audio = torch.FloatTensor(audio.astype(np.float32))
        
        if sampling_rate != self.sampling_rate:
            raise ValueError("{} {} SR doesn't match target {} SR".format(
                sampling_rate, self.sampling_rate))
            
        audio_norm = audio / self.max_wav_value
        audio_norm = audio_norm.unsqueeze(0)
        spec = spectrogram_torch(audio_norm, self.filter_length,
            self.sampling_rate, self.hop_length, self.win_length,
            center=False)
        spec = torch.squeeze(spec, 0)
        return spec, audio_norm
            
    def get_text(self, text):
        text_norm = text_to_sequence(text)
        text_norm = torch.LongTensor(text_norm)
        return text_norm

    def __getitem__(self, index):
        text = self.get_text(self.text[index])
        spec, audio_norm = self.get_audio(self.audio_path[index])
        return [text, spec, audio_norm]
    
    def play_audio(self, index):
        audio_path = self.audio_path[index]
        text = self.text[index]
        audio_path = data_path + '/wavs/'+ audio_path + '.wav'
        audio, sr = torchaudio.load(audio_path)
        return Audio(audio.numpy(), rate=sr)
    
    def __len__(self):
        return len(self.audio_path)
    
train_dataset = TextAudioLoader(metadata_df)
collate_fn = TextAudioCollate()

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    pin_memory=True,
    collate_fn=collate_fn,
    num_workers=4,
    persistent_workers=True,
)

In [ ]:
def get_padding(kernel_size, dilation=1):
    return int((kernel_size*dilation - dilation)/2)

class DiscriminatorP(torch.nn.Module):
    def __init__(self, period, kernel_size=5, stride=3, use_spectral_norm=False):
        super(DiscriminatorP, self).__init__()
        self.period = period
        self.use_spectral_norm = use_spectral_norm
        norm_f = weight_norm if use_spectral_norm == False else spectral_norm
        self.convs = nn.ModuleList([
            norm_f(Conv2d(1, 32, (kernel_size, 1), (stride, 1), padding=(get_padding(kernel_size, 1), 0))),
            norm_f(Conv2d(32, 128, (kernel_size, 1), (stride, 1), padding=(get_padding(kernel_size, 1), 0))),
            norm_f(Conv2d(128, 512, (kernel_size, 1), (stride, 1), padding=(get_padding(kernel_size, 1), 0))),
            norm_f(Conv2d(512, 1024, (kernel_size, 1), (stride, 1), padding=(get_padding(kernel_size, 1), 0))),
            norm_f(Conv2d(1024, 1024, (kernel_size, 1), 1, padding=(get_padding(kernel_size, 1), 0))),
        ])
        self.conv_post = norm_f(Conv2d(1024, 1, (3, 1), 1, padding=(1, 0)))

    def forward(self, x):
        fmap = []

        # 1d to 2d
        b, c, t = x.shape
        if t % self.period != 0: # pad first
            n_pad = self.period - (t % self.period)
            x = F.pad(x, (0, n_pad), "reflect")
            t = t + n_pad
        x = x.view(b, c, t // self.period, self.period)

        for l in self.convs:
            x = l(x)
            x = F.leaky_relu(x, 0.1)
            fmap.append(x)
        x = self.conv_post(x)
        fmap.append(x)
        x = torch.flatten(x, 1, -1)

        return x, fmap

class DiscriminatorS(torch.nn.Module):
    def __init__(self, use_spectral_norm=False):
        super(DiscriminatorS, self).__init__()
        norm_f = weight_norm if use_spectral_norm == False else spectral_norm
        self.convs = nn.ModuleList([
            norm_f(Conv1d(1, 16, 15, 1, padding=7)),
            norm_f(Conv1d(16, 64, 41, 4, groups=4, padding=20)),
            norm_f(Conv1d(64, 256, 41, 4, groups=16, padding=20)),
            norm_f(Conv1d(256, 1024, 41, 4, groups=64, padding=20)),
            norm_f(Conv1d(1024, 1024, 41, 4, groups=256, padding=20)),
            norm_f(Conv1d(1024, 1024, 5, 1, padding=2)),
        ])
        self.conv_post = norm_f(Conv1d(1024, 1, 3, 1, padding=1))

    def forward(self, x):
        fmap = []

        for l in self.convs:
            x = l(x)
            x = F.leaky_relu(x, 0.1)
            fmap.append(x)
        x = self.conv_post(x)
        fmap.append(x)
        x = torch.flatten(x, 1, -1)

        return x, fmap

class MultiPeriodDiscriminator(torch.nn.Module):
    def __init__(self, use_spectral_norm=False):
        super(MultiPeriodDiscriminator, self).__init__()
        periods = [2,3,5,7,11]

        discs = [DiscriminatorS(use_spectral_norm=use_spectral_norm)]
        discs = discs + [DiscriminatorP(i, use_spectral_norm=use_spectral_norm) for i in periods]
        self.discriminators = nn.ModuleList(discs)

    def forward(self, y, y_hat):
        y_d_rs = []
        y_d_gs = []
        fmap_rs = []
        fmap_gs = []
        for i, d in enumerate(self.discriminators):
            y_d_r, fmap_r = d(y)
            y_d_g, fmap_g = d(y_hat)
            y_d_rs.append(y_d_r)
            y_d_gs.append(y_d_g)
            fmap_rs.append(fmap_r)
            fmap_gs.append(fmap_g)

        return y_d_rs, y_d_gs, fmap_rs, fmap_gs

In [ ]:
class AudioSyn(nn.Module):
    def __init__(
        self, 
        n_vocab, 
        inter_channels, 
        hidden_channels, 
        filter_channels, 
        spec_channels
    ):
        super().__init__()

        self.enc_p = TextEncoder(
            vocab_size=n_vocab,
            hidden_channels=hidden_channels,
            latent_channels=inter_channels,
            feedforward_channels=filter_channels,
            padding_id=0,
        )
        
        self.seg_size = 8192 // 256
        self.dec = Decoder()
        self.enc_q = PosteriorEncoder(spec_channels)

        self.flow = ResidualCouplingBlock(
            channels=inter_channels,
            hidden_channels=hidden_channels,
        )
        self.dp = DurationPredictor(hidden_channels)
    
    def forward(self, x, x_lengths, y, y_lengths):
        g = None
        x, m_p, logs_p, x_mask = self.enc_p(x, x_lengths)
        z, m_q, logs_q, y_mask = self.enc_q(y, y_lengths)
        z_p = self.flow(z, y_mask, g=g)
        
        with torch.no_grad():
            # negative cross-entropy
            s_p_sq_r = torch.exp(-2 * logs_p) # [b, d, t]
            neg_cent1 = torch.sum(-0.5 * math.log(2 * math.pi) - logs_p, [1], keepdim=True) # [b, 1, t_s]
            neg_cent2 = torch.matmul(-0.5 * (z_p ** 2).transpose(1, 2), s_p_sq_r) # [b, t_t, d] x [b, d, t_s] = [b, t_t, t_s]
            neg_cent3 = torch.matmul(z_p.transpose(1, 2), (m_p * s_p_sq_r)) # [b, t_t, d] x [b, d, t_s] = [b, t_t, t_s]
            neg_cent4 = torch.sum(-0.5 * (m_p ** 2) * s_p_sq_r, [1], keepdim=True) # [b, 1, t_s]
            neg_cent = neg_cent1 + neg_cent2 + neg_cent3 + neg_cent4
            
            attn_mask = torch.unsqueeze(x_mask, 2) * torch.unsqueeze(y_mask, -1)
            attn = monotonic_align.maximum_path(neg_cent, attn_mask.squeeze(1)).unsqueeze(1).detach()

        w = attn.sum(2)
        logw_ = torch.log(w + 1e-6) * x_mask
        logw = self.dp(x, x_mask)
        l_length = torch.sum((logw - logw_)**2, [1,2]) / torch.sum(x_mask) # for averaging 
        
        # expand prior
        m_p = torch.matmul(attn.squeeze(1), m_p.transpose(1, 2)).transpose(1, 2)
        logs_p = torch.matmul(attn.squeeze(1), logs_p.transpose(1, 2)).transpose(1, 2)
    
        z_slice, ids_slice = slice_segments(z, x_lengths = y_lengths, segment_size = self.seg_size)
        o = self.dec(z_slice)
        return o, l_length, attn, ids_slice, x_mask, y_mask, (z, z_p, m_p, logs_p, m_q, logs_q)
    
    def infer(self, x, x_lengths, sid=None, noise_scale=1, length_scale=1, noise_scale_w=1., max_len=None):
        x, m_p, logs_p, x_mask = self.enc_p(x, x_lengths)
 
        g = None
        logw = self.dp(x, x_mask)
        
        w = torch.exp(logw) * x_mask * length_scale
        w_ceil = torch.ceil(w)
        y_lengths = torch.clamp_min(torch.sum(w_ceil, [1, 2]), 1).long()
        y_mask = torch.unsqueeze(sequence_mask(y_lengths, None), 1).to(x_mask.dtype)
        attn_mask = torch.unsqueeze(x_mask, 2) * torch.unsqueeze(y_mask, -1)
        attn = generate_path(w_ceil, attn_mask)

        m_p = torch.matmul(attn.squeeze(1), m_p.transpose(1, 2)).transpose(1, 2) # [b, t', t], [b, t, d] -> [b, d, t']
        logs_p = torch.matmul(attn.squeeze(1), logs_p.transpose(1, 2)).transpose(1, 2) # [b, t', t], [b, t, d] -> [b, d, t']

        z_p = m_p + torch.randn_like(m_p) * torch.exp(logs_p) * noise_scale
        z = self.flow(z_p, y_mask, g=g, reverse=True)
        o = self.dec((z * y_mask)[:,:,:max_len])
        return o, attn, y_mask, (z, z_p, m_p, logs_p)
        

In [ ]:
def feature_loss(fmap_r, fmap_g):
    loss = 0
    for dr, dg in zip(fmap_r, fmap_g):
        for rl, gl in zip(dr, dg):
            rl = rl.float().detach()
            gl = gl.float()
            loss += torch.mean(torch.abs(rl - gl))
    return loss * 2 

def discriminator_loss(disc_real_outputs, disc_generated_outputs):
    loss = 0
    r_losses = []
    g_losses = []
    for dr, dg in zip(disc_real_outputs, disc_generated_outputs):
        dr = dr.float()
        dg = dg.float()
        r_loss = torch.mean((1-dr)**2)
        g_loss = torch.mean(dg**2)
        loss += (r_loss + g_loss)
        r_losses.append(r_loss.item())
        g_losses.append(g_loss.item())

    return loss, r_losses, g_losses

def generator_loss(disc_outputs):
    loss = 0
    gen_losses = []
    for dg in disc_outputs:
        dg = dg.float()
        l = torch.mean((1-dg)**2)
        gen_losses.append(l)
        loss += l

    return loss, gen_losses

def kl_loss(z_p, logs_q, m_p, logs_p, z_mask):
    """
    z_p, logs_q: [b, h, t_t]
    m_p, logs_p: [b, h, t_t]
    """
    z_p = z_p.float()
    logs_q = logs_q.float()
    m_p = m_p.float()
    logs_p = logs_p.float()
    z_mask = z_mask.float()

    kl = logs_p - logs_q - 0.5
    kl += 0.5 * ((z_p - m_p)**2) * torch.exp(-2. * logs_p)
    kl = torch.sum(kl * z_mask)
    l = kl / torch.sum(z_mask)
    return l

In [ ]:
class VitTrainer:
    def __init__(self, g, d, data_loader, total_steps, save_n_steps, \
                 ckpt_dir, LR, segment_size, device='cuda', load_path = None):
        super().__init__()
        self.g = g.to(device)
        self.d = d.to(device)
        
        self.data_loader = data_loader
        self.optim_g = torch.optim.AdamW(self.g.parameters(), LR, betas=[0.8, 0.99], eps=1e-9)                 
        self.optim_d = torch.optim.AdamW(self.d.parameters(), LR, betas=[0.8, 0.99], eps=1e-9)
        self.scaler = torch.amp.GradScaler("cuda", enabled=True)
        
        self.step = 1
        self.ckpt_dir = ckpt_dir
        self.device = device
        self.total_steps = total_steps
        self.save_n_steps = save_n_steps
        
        if load_path is not None:
            self.load_state_dict(load_path)
            print("sucessful load state dict !!!!!!")
            print(f"start from step {self.step}")
        
    def state_dict(self, step):
        return {
            "step": step,
            'g': self.g.state_dict(),
            'd': self.d.state_dict()
        }
    
    def load_state_dict(self, path):
        state_dict = torch.load(path)
        self.g.load_state_dict(state_dict['g'])
        self.d.load_state_dict(state_dict['d'])
        self.step = state_dict['step']
        
    def train(self):
        start = time.time()
        data_iter = iter(self.data_loader)
        print(f'Start of step {self.step}')
        
        for step in trange(
            self.step,
            self.total_steps + 1,
            desc="Training progress",
        ):
            try:
                batch = next(data_iter)
            except StopIteration:
                data_iter = iter(self.data_loader)
                batch = next(data_iter)

            x, x_lengths, spec, spec_lengths, y, y_lengths = [
                tensor.to(self.device, non_blocking=True)
                for tensor in batch
            ]
            
            with torch.amp.autocast("cuda", dtype=torch.float16, enabled=True):
                y_hat, l_length, attn, ids_slice, x_mask, z_mask,\
                (z, z_p, m_p, logs_p, m_q, logs_q) = self.g(x, x_lengths, spec, spec_lengths)
                mel = spec_to_mel_torch(spec, 1024, 80, 22050, 0.0, None)
                y_mel, _ = slice_segments(mel, ids_str = ids_slice, segment_size = segment_size // 256)
                y_hat_mel = mel_spectrogram_torch(y_hat.squeeze(1), 1024, 80, 22050, 256, 1024, 0.0, None)
                y, _ = slice_segments(y, ids_str = ids_slice * 256, segment_size = segment_size) 

                # Discriminator
                y_d_hat_r, y_d_hat_g, _, _ = self.d(y, y_hat.detach())
                with torch.amp.autocast("cuda", dtype=torch.float16, enabled=False):
                    loss_disc, losses_disc_r, losses_disc_g = discriminator_loss(y_d_hat_r, y_d_hat_g)
                    loss_disc_all = loss_disc
        
            self.optim_d.zero_grad()
            self.scaler.scale(loss_disc_all).backward()
            self.scaler.unscale_(self.optim_d)
            self.grad_norm = nn.utils.clip_grad_norm_(self.d.parameters(), 1e9)
            self.scaler.step(self.optim_d)

            with torch.amp.autocast("cuda", dtype=torch.float16, enabled=True):
                y_d_hat_r, y_d_hat_g, fmap_r, fmap_g = self.d(y, y_hat)
                with torch.amp.autocast("cuda", dtype=torch.float16, enabled=False):
                    loss_dur = torch.sum(l_length.float())
                    loss_mel = F.l1_loss(y_mel, y_hat_mel) * 45
                    loss_kl = kl_loss(z_p, logs_q, m_p, logs_p, z_mask)

                    loss_fm = feature_loss(fmap_r, fmap_g)
                    loss_gen, losses_gen = generator_loss(y_d_hat_g)
                    loss_gen_all = loss_gen + loss_fm + loss_mel + loss_dur + loss_kl
                    
            self.optim_g.zero_grad()
            self.scaler.scale(loss_gen_all).backward()
            self.scaler.unscale_(self.optim_g)
            self.grad_norm = nn.utils.clip_grad_norm_(self.g.parameters(), 1e9)
            self.scaler.step(self.optim_g)
            self.scaler.update()
            
            if step % self.save_n_steps == 0:
                epoch = step // self.save_n_steps
                time_minutes = (time.time() - start) / 60
                torch.save(self.state_dict(step), f"{self.ckpt_dir}/weight_tts_epoch{epoch}.pt")
                
                print(f"epoch: {epoch}, d_loss: {loss_disc_all.item()} ~~~~~~")
                print(f"epoch: {epoch}, g_loss: {loss_gen_all.item()} ~~~~~~")
                print (f'Time taken for epoch {epoch} is {time_minutes:.3f} min\n') 
                print(f"sucessful saving epoch {epoch} state dict !!!!!!!")
                start = time.time()
                self.generate(epoch) 
    
    @torch.no_grad()
    def generate(self, epochs):
        sample_rate = 22050
        hop_length = 256
        max_len = 1000

        output_dir = (Path(self.ckpt_dir) / "audio" / f"epoch_{epochs:04d}")
        output_dir.mkdir(parents=True, exist_ok=True)
        was_training = self.g.training
        self.g.eval()

        try:
            dataset = self.data_loader.dataset
            # Use the same two samples to compare different epochs.
            for index in range(min(2, len(dataset))):
                text, _, waveform = dataset[index]

                x = text.unsqueeze(0).to(self.device)
                x_lengths = torch.tensor(
                    [text.numel()],
                    dtype=torch.long,
                    device=self.device,
                )
                y_hat, _, mask, *_ = self.g.infer(x, x_lengths, max_len=max_len)

                # Limit playback to the actual generated waveform length.
                predicted_frames = int(mask[0].sum().item())
                generated_length = min(
                    predicted_frames * hop_length,
                    y_hat.size(-1),
                )
                gt = waveform.squeeze(0).detach().float().cpu().numpy()
                    
                generated = (
                    y_hat[0, 0, :generated_length]
                    .detach().float().cpu().numpy()
                )
                gt_path = output_dir / f"sample_{index + 1}_gt.wav"
                gen_path = (output_dir / f"sample_{index + 1}_generated.wav")

                # Save WAV files without volume normalization.
                sf.write(gt_path, gt, sample_rate, subtype="PCM_16")
                sf.write(gen_path, generated, sample_rate, subtype="PCM_16")

                print(f"\nEpoch {epochs} | Sample {index + 1}")

                # Play the saved files directly in the notebook.
                print(f"GT — {len(gt) / sample_rate:.2f} seconds")
                print(f"Saved: {gt_path}")
                display(Audio(filename=str(gt_path)))

                print(
                    f"Generated — "
                    f"{len(generated) / sample_rate:.2f} seconds"
                )
                print(f"Saved: {gen_path}")
                display(Audio(filename=str(gen_path)))

                if predicted_frames > max_len:
                    print(
                        f"Note: generated audio was truncated "
                        f"to {max_len} frames."
                    )
        finally:
            self.g.train(was_training)

In [ ]:
n_vocab = len(symbols)
spec_channels = 1024 // 2 + 1
segment_size = 8192
hop_length = 256

inter_channels = 192
hidden_channels = 192
filter_channels = 768
total_steps = 40000
save_n_steps = 8000

ckpt_dir = './model_weight/'
os.makedirs(ckpt_dir, exist_ok=True)
load_path = None

d_model = MultiPeriodDiscriminator()
g_model = AudioSyn(
    n_vocab=n_vocab, 
    inter_channels=inter_channels, 
    hidden_channels=hidden_channels, 
    filter_channels=filter_channels, 
    spec_channels=spec_channels
)  
    
trainer = VitTrainer(
    g=g_model, 
    d=d_model, 
    data_loader=train_loader, 
    total_steps=total_steps, 
    save_n_steps=save_n_steps, 
    ckpt_dir=ckpt_dir, 
    LR=2e-4, 
    segment_size=segment_size
)

In [ ]:
trainer.train()